# COMP5329 — Deep Learning

**Tutorial 11 — Self-Supervised Representation Learning: From Autoencoders to CLIP**

**Semester 1, 2026**

### Learning Objectives

By the end of this tutorial you will be able to:

1. Explain **self-supervised learning** as a paradigm and distinguish generative, contrastive, and non-contrastive approaches.
2. Implement an **Autoencoder** from scratch and visualise how the bottleneck learns compressed representations.
3. Explain how **Masked Autoencoders (MAE)** extend the AE idea to Vision Transformers, connecting to BERT's MLM from Week 8.
4. Derive the **InfoNCE loss** and implement it, understanding the role of temperature $\tau$.
5. Implement the **SimCLR** pipeline from scratch -- augmentation, encoding, projection, contrastive loss.
6. Explain how **MoCo**'s momentum encoder and queue decouple batch size from the number of negatives.
7. Explain why **BYOL** and **SimSiam** do not collapse without negatives, identifying the specific anti-collapse mechanism in each.
8. Describe **DINO**'s self-distillation framework and how centering prevents mode collapse.
9. Connect **CLIP** (from Week 9) to the contrastive SSL framework, identifying it as cross-modal InfoNCE.
10. Compare all nine methods across supervision signal, negatives, anti-collapse mechanism, and downstream capability.

### Topic Coverage

Week 11 covers **self-supervised representation learning: from autoencoders to CLIP**. The full topic list (see `Week11_Self_Study_SSL.ipynb`) is:

- 📖 **Foundations — why SSL exists, learning without labels, the SSL landscape** *(self-study, Part I)*
- ✅ **Autoencoder (AE)** — bottleneck compression, from-scratch implementation *(tutorial)*
- ✅ **Masked Autoencoder (MAE)** — BERT for images, 75% masking, visible-patch-only encoder *(tutorial, conceptual)*
- 📖 **Why the classic AE programme plateaued and how MAE revived it** *(self-study, Part II)*
- ✅ **InfoNCE loss** — contrastive learning core, temperature τ, mutual-information bound *(tutorial)*
- ✅ **SimCLR** — augmentation pipeline, NT-Xent loss, symmetric contrastive loss *(tutorial, with in-class practice)*
- ✅ **MoCo** — momentum encoder + queue, decoupling negatives from batch size *(tutorial)*
- ✅ **Non-contrastive methods — BYOL, SimSiam, DINO** — anti-collapse without negatives *(tutorial, briefly; deeper in self-study Part IV)*
- 📖 **The asymmetry principle as a unified lens** across non-contrastive methods *(self-study, Part IV)*
- ✅ **CLIP as cross-modal InfoNCE** — the SSL lens on Week 9's CLIP *(tutorial, briefly)*
- 📖 **Three philosophies on one slide — generative vs contrastive vs non-contrastive synthesis** *(self-study, Parts V–VI)*

Due to time constraints, the tutorial focuses on **one unifying lens — every SSL method = (pretext task) + (anti-collapse mechanism)** — and works through one in-class implementation (NT-Xent). BYOL and DINO are covered briefly; the full asymmetry-principle framing, the plateau/revival story for generative SSL, and the Part VI synthesis chapters are left as self-study.

The live session is organised into three parts: **Part A** — tutor walkthrough, **Part B** — in-class coding exercise, **Part C** — exam-style Q&A.

---
# Part A · Tutor Walkthrough

## 0. The Story So Far and Where We Are Going

In Week 8 we saw that BERT's Masked Language Model pre-trains on unlabelled text -- this IS self-supervised learning. In Week 9 we saw CLIP align images and text without classification labels. This week we systematically study the **self-supervised representation learning** landscape.

The central question: **How do we learn good representations WITHOUT labels?**

Four progressively deeper answers:

| Phase | Methods | Core Question | Code Depth |
|---|---|---|---|
| 1: Generative | AE, MAE | What if the label IS the input? | AE from scratch; MAE conceptual |
| 2: Contrastive | InfoNCE, SimCLR, MoCo | What if we learn by comparison? | InfoNCE + SimCLR from scratch |
| 3: Non-Contrastive | BYOL, SimSiam, DINO | Do we even need negatives? | Conceptual + collapse demo |
| 4: Multi-Modal | CLIP | Can we align across modalities? | Light code (revisit Week 9) |

**Three evolution lines:**
- **Negatives**: None (AE) $\to$ large batch (SimCLR) $\to$ queue (MoCo) $\to$ none again (BYOL/SimSiam) $\to$ cross-modal (CLIP)
- **Anti-collapse**: Bottleneck (AE) $\to$ repulsion (contrastive) $\to$ predictor+momentum (BYOL) $\to$ stop-gradient (SimSiam) $\to$ centering (DINO)
- **Signal source**: Pixels (AE) $\to$ augmentation invariance (SimCLR-DINO) $\to$ cross-modal correspondence (CLIP)

---

## Phase 1: Generative Self-Supervision

### 1.1 Autoencoder (AE)

The simplest self-supervised idea: **the label IS the input itself**.

- **Encoder** $f_\theta: \mathbb{R}^d \to \mathbb{R}^k$ with $k \ll d$ (bottleneck)
- **Decoder** $g_\phi: \mathbb{R}^k \to \mathbb{R}^d$ (reconstruction)
- **Loss**: $\mathcal{L}_{\text{AE}} = \frac{1}{N}\sum_{i=1}^{N} \|x_i - g_\phi(f_\theta(x_i))\|^2$

The bottleneck $k \ll d$ forces **compression** -- the encoder must learn the most important features. Without the bottleneck, the network could learn the identity function (useless).

Connection: a linear autoencoder recovers PCA. A nonlinear one learns a more expressive manifold.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────────
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T

%matplotlib inline

In [ ]:
# ── Autoencoder from scratch ─────────────────────────────────────────────────
# Synthetic data: 2D points on a circle, embedded in 20D via random projection + noise
torch.manual_seed(42)
N = 2000
angles = torch.rand(N) * 2 * math.pi
circle_2d = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1)  # (N, 2)
proj = torch.randn(2, 20)  # random projection to 20D
data_20d = circle_2d @ proj + torch.randn(N, 20) * 0.1  # (N, 20)

class Autoencoder(nn.Module):
    def __init__(self, input_dim=20, hidden_dim=64, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim))
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim))
    def encode(self, x): return self.encoder(x)
    def decode(self, z): return self.decoder(z)
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z

ae = Autoencoder()
opt_ae = optim.Adam(ae.parameters(), lr=1e-3)
for epoch in range(300):
    idx = torch.randperm(N)[:256]
    x_hat, z = ae(data_20d[idx])
    loss = F.mse_loss(x_hat, data_20d[idx])
    opt_ae.zero_grad(); loss.backward(); opt_ae.step()
    if (epoch+1) % 100 == 0: print(f'  AE epoch {epoch+1}, loss: {loss.item():.4f}')

In [ ]:
# ── AE visualisation ─────────────────────────────────────────────────────────
with torch.no_grad():
    _, z_all = ae(data_20d)
    x_recon, _ = ae(data_20d[:5])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
# Latent space coloured by original angle
sc = ax1.scatter(z_all[:, 0].numpy(), z_all[:, 1].numpy(),
                  c=angles.numpy(), cmap='hsv', s=3, alpha=0.6)
ax1.set_title('Learned Latent Space (coloured by angle)')
ax1.set_xlabel('z1'); ax1.set_ylabel('z2')
plt.colorbar(sc, ax=ax1, label='Original angle')

# Reconstruction quality
ax2.bar(range(20), data_20d[0].numpy(), alpha=0.6, label='Original')
ax2.bar(range(20), x_recon[0].numpy(), alpha=0.6, label='Reconstructed')
ax2.set_title('Reconstruction: Original vs Decoded')
ax2.legend()
plt.tight_layout(); plt.show()
print('The bottleneck recovers the circular structure from 20D data!')

### 1.2 Masked Autoencoder (MAE)

MAE (He et al., 2022) applies the autoencoder idea to Vision Transformers -- it is **"BERT for images"**:

1. Split image into non-overlapping patches (like ViT)
2. **Randomly mask 75%** of patches
3. Encoder (ViT) processes **only visible patches** -- this is the efficiency trick
4. Lightweight decoder reconstructs the **masked patches**

**Loss**: MSE only on masked positions: $\mathcal{L}_{\text{MAE}} = \frac{1}{|M|}\sum_{i \in M}\|p_i - \hat{p}_i\|^2$

| Aspect | BERT MLM (Week 8) | MAE |
|---|---|---|
| Modality | Text | Image |
| Masking ratio | 15% | 75% |
| Why different? | Language is information-dense | Images have high spatial redundancy |
| Encoder input | All tokens (with `[MASK]`) | Visible patches only |

> **Transition**: Autoencoders learn by reconstructing the input. But reconstruction is a **pixel-level** objective -- it forces the model to memorise low-level details (textures, exact values) rather than high-level semantics (objects, relationships). A cat rotated 5° has nearly identical semantics but very different pixels. Can we learn representations that capture **what** is in an image rather than its exact appearance? Instead of reconstructing, let us **compare**.

---

## Phase 2: Contrastive Learning

### 2.1 The InfoNCE Loss

The foundational loss for contrastive learning. Given a query $q$, one positive key $k^+$, and $K$ negative keys $\{k^-_j\}$:

$$\mathcal{L}_{\text{InfoNCE}} = -\log \frac{\exp(\text{sim}(q, k^+)/\tau)}{\exp(\text{sim}(q, k^+)/\tau) + \sum_{j=1}^{K}\exp(\text{sim}(q, k^-_j)/\tau)}$$

This is just **(K+1)-way cross-entropy** where the positive is the "correct class" and negatives are "incorrect classes".

**Temperature** $\tau$:
- $\tau \to 0$: hard assignment (winner-take-all) -- very peaked softmax
- $\tau \to \infty$: uniform distribution -- no discrimination
- Typical: $\tau = 0.07$ to $0.5$

**Connection to mutual information**: InfoNCE is a lower bound on $I(q; k^+)$. More negatives = tighter bound.

In [ ]:
# ── InfoNCE loss from scratch ───────────────────────────────────────────────

def info_nce_loss(query, positive, negatives, temperature=0.1):
    """InfoNCE contrastive loss.
    Args:
        query:     (D,) query embedding
        positive:  (D,) positive key embedding
        negatives: (K, D) negative key embeddings
        temperature: scalar
    Returns: scalar loss.
    """
    pos_sim = torch.dot(query, positive) / temperature
    neg_sim = torch.matmul(negatives, query) / temperature  # (K,)
    logits = torch.cat([pos_sim.unsqueeze(0), neg_sim])     # (1+K,)
    labels = torch.tensor(0)  # positive is at index 0
    return F.cross_entropy(logits, labels)

# Verification
torch.manual_seed(42)
D = 128
q = F.normalize(torch.randn(D), dim=0)
k_pos = F.normalize(q + torch.randn(D) * 0.1, dim=0)  # similar to query
k_neg = F.normalize(torch.randn(50, D), dim=1)          # random negatives

loss = info_nce_loss(q, k_pos, k_neg, temperature=0.1)
print(f'InfoNCE loss (positive similar to query): {loss.item():.4f}')
loss_hard = info_nce_loss(q, F.normalize(torch.randn(D), dim=0), k_neg, 0.1)
print(f'InfoNCE loss (random positive):           {loss_hard.item():.4f}')
print('Lower loss when positive is truly similar!')

In [ ]:
# ── Temperature effect visualisation ───────────────────────────────────────
temps = [0.01, 0.07, 0.1, 0.5, 1.0, 5.0]
pos_sims = torch.linspace(-1, 1, 200)

fig, ax = plt.subplots(figsize=(8, 5))
for tau in temps:
    losses = []
    for s in pos_sims:
        # Fixed negatives with similarity ~0
        logits = torch.cat([s.unsqueeze(0)/tau, torch.zeros(10)/tau])
        losses.append(F.cross_entropy(logits.unsqueeze(0), torch.tensor([0])).item())
    ax.plot(pos_sims.numpy(), losses, label=f'τ={tau}')
ax.set_xlabel('Positive Similarity'); ax.set_ylabel('InfoNCE Loss')
ax.set_title('Effect of Temperature on InfoNCE Loss')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('Lower τ = sharper gradient around the decision boundary.')

### 2.2 SimCLR: A Simple Framework for Contrastive Learning

SimCLR (Chen et al., 2020) applies InfoNCE to visual representation learning:

1. Sample a minibatch of $N$ images
2. Apply **two random augmentations** to each image $\to$ $2N$ views
3. Encode each view with backbone $f$ (e.g., ResNet): $h_i = f(\tilde{x}_i)$
4. Project with MLP $g$: $z_i = g(h_i)$
5. Apply **symmetric InfoNCE**: for each image, its two views are the positive pair; all other $2(N-1)$ views are negatives

**Key finding**: Representations **before** the projection head ($h$) are better for downstream tasks than those after ($z$). The projection head discards task-specific information that helps contrastive learning but hurts transfer.

**Augmentations matter enormously**: Random crop + resize, colour distortion, Gaussian blur. The composition defines what invariances the model learns.

**Limitation**: Needs **large batch sizes** (4096-8192) -- more negatives = better InfoNCE bound.

---
# Part B · In-Class Coding Exercise

## NT-Xent from Scratch

You have just seen the SimCLR pipeline. Its mathematical heart is the **NT-Xent (Normalised Temperature-scaled Cross-Entropy)** loss — the same loss that powers MoCo and CLIP. In this exercise you will implement it yourself on **synthetic embeddings** (no GPU / no CIFAR needed).

### Setup

You are given two batches of L2-normalised projections from the same $B$ images under two augmentations:
- `z1`: shape `(B, D)` — view 1 embeddings
- `z2`: shape `(B, D)` — view 2 embeddings

For sample $i$, the **positive** is `z2[i]` (the other view of the same image). All other $2B - 2$ embeddings in the combined batch are **negatives**.

### Task — fill in the 3 TODOs

```python
def nt_xent_loss_student(z1, z2, temperature=0.5):
    # TODO 1: Build the (2B, D) stacked tensor and the (2B, 2B) cosine-sim matrix scaled by 1/τ
    # TODO 2: Mask out the diagonal (a sample is its own most similar — exclude it)
    # TODO 3: Build the label vector that points to the correct positive for each row,
    #         then return F.cross_entropy(logits, labels)
```

After implementing, run the **3 sanity checks** below:

1. **Identity sanity:** when `z1 == z2`, every row's positive sits exactly on top of itself in cosine space → loss should be small.
2. **Random sanity:** when `z1` and `z2` are independent random unit vectors, loss should be close to `log(2B-1)`.
3. **Temperature sweep:** plot loss vs τ for τ ∈ {0.05, 0.1, 0.5, 1.0, 5.0} on the *same* random data. Explain in one sentence why both extremes are bad.

> 💡 **Hint:** You may compare against the reference `nt_xent_loss_reference` defined a few cells below — but try yourself first!

In [ ]:
# ── Student skeleton: NT-Xent from scratch ────────────────────────────────
# Run this cell after filling in the 3 TODOs.

import torch
import torch.nn.functional as F

def nt_xent_loss_student(z1, z2, temperature=0.5):
    """Your implementation of NT-Xent.
    Args:
        z1, z2: (B, D) L2-normalised projections of two augmented views.
        temperature: scalar τ.
    Returns:
        scalar loss (torch.Tensor).
    """
    B, D = z1.shape

    # TODO 1: Stack z1, z2 into z of shape (2B, D), then build the
    #         (2B, 2B) similarity matrix sim = (z @ z.T) / temperature.
    z = ...
    sim = ...

    # TODO 2: Remove the diagonal (each row's self-similarity).
    #         After this step `sim` should have shape (2B, 2B - 1).
    mask = ~torch.eye(2 * B, dtype=torch.bool)
    sim = ...

    # TODO 3: Build the labels vector pointing to the correct positive.
    #         Hint: row i in [0, B) has its positive at column (B - 1 + i) of
    #         the masked matrix; row i in [B, 2B) has its positive at column i - B.
    labels = ...

    return F.cross_entropy(sim, labels)


# ── Sanity check 1: identical views → small loss ──────────────────────────
torch.manual_seed(0)
B, D = 8, 32
z = F.normalize(torch.randn(B, D), dim=1)
loss_id = nt_xent_loss_student(z, z.clone(), temperature=0.5)
print(f"Identity loss (should be small): {loss_id.item():.4f}")

# ── Sanity check 2: random independent views → ~log(2B-1) ─────────────────
import math
z1 = F.normalize(torch.randn(B, D), dim=1)
z2 = F.normalize(torch.randn(B, D), dim=1)
loss_rand = nt_xent_loss_student(z1, z2, temperature=0.5)
print(f"Random loss (≈ log(2B-1) = {math.log(2*B-1):.4f}): {loss_rand.item():.4f}")

# ── Sanity check 3: temperature sweep ─────────────────────────────────────
import matplotlib.pyplot as plt
taus = [0.05, 0.1, 0.5, 1.0, 5.0]
losses = [nt_xent_loss_student(z1, z2, temperature=t).item() for t in taus]
plt.figure(figsize=(6, 4))
plt.semilogx(taus, losses, marker='o')
plt.xlabel('temperature τ'); plt.ylabel('NT-Xent loss')
plt.title('Effect of temperature on random embeddings')
plt.grid(True, alpha=0.3); plt.show()

# Q: Why is τ → 0 bad? Why is τ → ∞ bad? (Discuss in one sentence each.)


### ✅ Reference Solution

Try the TODOs yourself first. The cell below is the tutor's reference — it should match the existing `nt_xent_loss` used by the SimCLR training pipeline that follows.


In [ ]:
# ── Reference solution ────────────────────────────────────────────────────

def nt_xent_loss_reference(z1, z2, temperature=0.5):
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)                          # (2B, D)
    sim = torch.matmul(z, z.T) / temperature                 # (2B, 2B)
    mask = ~torch.eye(2 * B, dtype=torch.bool)
    sim = sim.masked_select(mask).view(2 * B, 2 * B - 1)    # (2B, 2B - 1)
    labels = torch.cat([torch.arange(B - 1, 2 * B - 1),
                        torch.arange(0, B)])                 # (2B,)
    return F.cross_entropy(sim, labels)


# Sanity-check the reference matches the student version on the same inputs:
torch.manual_seed(0)
B, D = 8, 32
z1 = F.normalize(torch.randn(B, D), dim=1)
z2 = F.normalize(torch.randn(B, D), dim=1)
print(f"Reference loss @ τ=0.5: {nt_xent_loss_reference(z1, z2, 0.5).item():.4f}")

# Discussion (tutor walk-through):
# • τ → 0 : softmax becomes one-hot → gradient vanishes for all but the single
#           hardest negative → unstable, very sensitive to noise.
# • τ → ∞ : softmax becomes uniform → loss saturates at log(2B-1) → no learning
#           signal, positives and negatives are indistinguishable.
# • Sweet spot for SimCLR (≈0.1–0.5) and CLIP (≈0.07) trades off these regimes.


In [ ]:
# ── SimCLR from scratch (CIFAR-10 subset) ─────────────────────────────

# Augmentation pipeline
simclr_transform = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

class TwoViewTransform:
    """Return two augmented views of the same image."""
    def __init__(self, transform):
        self.transform = transform
    def __call__(self, x):
        return self.transform(x), self.transform(x)

# Load a small subset of CIFAR-10
cifar = torchvision.datasets.CIFAR10(root='./data', train=True, download=True,
                                      transform=TwoViewTransform(simclr_transform))
# Use only 2000 samples for speed
subset_idx = list(range(2000))
cifar_subset = torch.utils.data.Subset(cifar, subset_idx)
loader = torch.utils.data.DataLoader(cifar_subset, batch_size=128, shuffle=True, drop_last=True)
print(f'Dataset size: {len(cifar_subset)}, Batches: {len(loader)}')

In [ ]:
# ── SimCLR model + NT-Xent loss ───────────────────────────────────────────

class SimpleCNN(nn.Module):
    """Lightweight backbone for CIFAR-10."""
    def __init__(self, feat_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        self.fc = nn.Linear(128, feat_dim)
    def forward(self, x):
        h = self.conv(x).squeeze(-1).squeeze(-1)  # (B, 128)
        return self.fc(h)                           # (B, feat_dim)

class SimCLRModel(nn.Module):
    def __init__(self, feat_dim=128, proj_dim=64):
        super().__init__()
        self.backbone = SimpleCNN(feat_dim)
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, feat_dim), nn.ReLU(),
            nn.Linear(feat_dim, proj_dim))
    def forward(self, x):
        h = self.backbone(x)      # representation (for downstream)
        z = self.projector(h)      # projection (for contrastive loss)
        return h, F.normalize(z, dim=1)

def nt_xent_loss(z1, z2, temperature=0.5):
    """NT-Xent loss (SimCLR). z1, z2: (B, D) L2-normalised."""
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)                         # (2B, D)
    sim = torch.matmul(z, z.T) / temperature                # (2B, 2B)
    # Mask out self-similarity
    mask = ~torch.eye(2*B, dtype=torch.bool)
    sim = sim.masked_select(mask).view(2*B, 2*B - 1)       # (2B, 2B-1)
    # Labels: positive pair is at position B-1 for first B, and at 0..B-1 for second B
    labels = torch.cat([torch.arange(B-1, 2*B-1), torch.arange(0, B)])  # (2B,)
    return F.cross_entropy(sim, labels)

In [ ]:
# ── SimCLR training ─────────────────────────────────────────────────────────
torch.manual_seed(42)
model = SimCLRModel()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
losses = []

for epoch in range(20):
    epoch_loss = 0
    for (x1, x2), _ in loader:  # _ = labels (not used!)
        _, z1 = model(x1)
        _, z2 = model(x2)
        loss = nt_xent_loss(z1, z2, temperature=0.5)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(loader)
    losses.append(avg)
    if (epoch+1) % 5 == 0:
        print(f'  SimCLR epoch {epoch+1}, loss: {avg:.3f}')

plt.figure(figsize=(7, 3))
plt.plot(losses); plt.xlabel('Epoch'); plt.ylabel('NT-Xent Loss')
plt.title('SimCLR Training (no labels used!)'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── SimCLR learned representations (t-SNE) ─────────────────────────────
# Extract representations (BEFORE projection head!) with true labels
eval_transform = T.Compose([T.ToTensor(), T.Normalize((0.4914,0.4822,0.4465),(0.247,0.243,0.261))])
eval_data = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=eval_transform)
eval_subset = torch.utils.data.Subset(eval_data, list(range(1000)))
eval_loader = torch.utils.data.DataLoader(eval_subset, batch_size=256, shuffle=False)

all_h, all_labels = [], []
model.eval()
with torch.no_grad():
    for x, y in eval_loader:
        h, _ = model(x)
        all_h.append(h)
        all_labels.append(y)
all_h = torch.cat(all_h).numpy()
all_labels = torch.cat(all_labels).numpy()

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
h_2d = tsne.fit_transform(all_h)

plt.figure(figsize=(8, 6))
classes = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']
for c in range(10):
    mask = all_labels == c
    plt.scatter(h_2d[mask, 0], h_2d[mask, 1], s=5, alpha=0.5, label=classes[c])
plt.legend(fontsize=7, markerscale=3)
plt.title('SimCLR Representations (t-SNE) -- Labels NEVER used in training!')
plt.tight_layout(); plt.show()
print('Representations cluster by semantic class even though no labels were used!')

### 2.3 MoCo: Momentum Contrast

SimCLR needs **large batch sizes** (4096+) because more in-batch negatives = better InfoNCE. MoCo decouples negatives from batch size using two innovations:

1. **Momentum encoder** $f_k$: a slowly-updated copy of the query encoder
$$\theta_k \leftarrow m \cdot \theta_k + (1 - m) \cdot \theta_q, \quad m = 0.999$$

2. **Queue**: A FIFO buffer of $K$ encoded keys (e.g., $K = 65{,}536$) from previous batches. Negatives come from the queue, not the current batch.

**Why momentum?** If the key encoder changed rapidly, old keys in the queue would be inconsistent with new ones. Momentum ensures slow, smooth evolution -- all keys in the queue are "almost" from the same encoder.

| Aspect | SimCLR | MoCo |
|---|---|---|
| Negatives | Current batch: $2(N-1)$ | Queue: $K$ (e.g., 65,536) |
| Batch size | Very large (4096+) | Normal (256) |
| Key encoder | Same as query | Momentum-updated |
| Extra memory | None | Queue + momentum encoder |

In [ ]:
# ── MoCo: momentum update + queue ─────────────────────────────────────────

@torch.no_grad()
def momentum_update(online_enc, momentum_enc, m=0.999):
    """Update momentum encoder: theta_k = m * theta_k + (1-m) * theta_q."""
    for p_o, p_m in zip(online_enc.parameters(), momentum_enc.parameters()):
        p_m.data = m * p_m.data + (1 - m) * p_o.data

class MoCoQueue:
    """FIFO queue of encoded keys for MoCo negatives."""
    def __init__(self, dim, max_size=4096):
        self.queue = F.normalize(torch.randn(max_size, dim), dim=1)
        self.ptr = 0
        self.max_size = max_size
    def enqueue(self, keys):
        B = keys.size(0)
        end = self.ptr + B
        if end <= self.max_size:
            self.queue[self.ptr:end] = keys.detach()
        else:  # wrap around
            overflow = end - self.max_size
            self.queue[self.ptr:] = keys[:B - overflow].detach()
            self.queue[:overflow] = keys[B - overflow:].detach()
        self.ptr = end % self.max_size

# Demo
queue = MoCoQueue(dim=64, max_size=1024)
print(f'Queue size: {queue.max_size}, shape: {queue.queue.shape}')
queue.enqueue(torch.randn(32, 64))  # add a batch
print(f'After enqueue: ptr = {queue.ptr}')
print('Queue provides a large pool of negatives without needing a large batch!')

> **Transition**: Contrastive methods (SimCLR, MoCo) work remarkably well, but they depend on **negatives** -- and managing negatives is expensive. SimCLR needs enormous batches; MoCo needs a queue and momentum encoder. A deeper question: are negatives **actually necessary**? What if we just align positive pairs? The danger: the model could **collapse** -- mapping everything to the same point gives perfect alignment with zero loss. Phase 3's methods solve this elegantly.

---

## Phase 3: Non-Contrastive Learning

### 3.1 BYOL (Bootstrap Your Own Latent)

> 📖 **Self-study section** — covered briefly in the tutor-led recap; full detail for home reading.

BYOL (Grill et al., 2020) learns with **only positive pairs -- no negatives at all**.

**Asymmetric architecture**:
- **Online network**: encoder $f_\theta$ + projector $g_\theta$ + **predictor** $q_\theta$
- **Target network**: encoder $f_\xi$ + projector $g_\xi$ (no predictor). Parameters $\xi$ are **momentum-updated**: $\xi \leftarrow m \cdot \xi + (1 - m) \cdot \theta$

**Loss** (cosine similarity between online prediction and target projection):

$$\mathcal{L}_{\text{BYOL}} = 2 - 2 \cdot \frac{\langle q_\theta(z_1),\; \bar{z}_2'\rangle}{\|q_\theta(z_1)\| \cdot \|\bar{z}_2'\|}$$

where $\bar{z}_2'$ means **stop-gradient** (target outputs are treated as fixed).

**Why doesn't it collapse?** Two mechanisms:
1. **Predictor** $q_\theta$: introduces asymmetry -- the online network must predict *beyond* what it knows
2. **Momentum update**: the target evolves slowly, providing a **stable regression target**

### 3.2 SimSiam: Simplicity is All You Need

SimSiam (Chen & He, 2021) goes even further: **no momentum encoder**!

- **Shared encoder** + **shared projector** + **predictor** (same weights for both branches)
- The only asymmetry: **stop-gradient** on one branch

$$\mathcal{L}_{\text{SimSiam}} = -\frac{1}{2}\left[\frac{p_1 \cdot \text{sg}(z_2)}{\|p_1\|\|\text{sg}(z_2)\|} + \frac{p_2 \cdot \text{sg}(z_1)}{\|p_2\|\|\text{sg}(z_1)\|}\right]$$

**Why stop-gradient prevents collapse**: Without it, both branches optimise to be identical $\to$ collapse. With stop-gradient, one branch acts as a **fixed target**, creating **alternating optimisation** (like EM): update the predictor to match the target, then the target implicitly shifts.

The following demo shows this dramatically:

In [ ]:
# ── SimSiam: stop-gradient collapse demo ───────────────────────────────
# Minimal experiment: train a projector+predictor on 2D data
# WITH vs WITHOUT stop-gradient

torch.manual_seed(42)
data = torch.randn(500, 2) * 2  # 2D data

def train_simsiam(use_stop_grad, steps=500):
    encoder = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 16))
    predictor = nn.Sequential(nn.Linear(16, 64), nn.ReLU(), nn.Linear(64, 16))
    opt = optim.Adam(list(encoder.parameters()) + list(predictor.parameters()), lr=1e-3)
    for _ in range(steps):
        x1 = data + torch.randn_like(data) * 0.3  # augmentation 1
        x2 = data + torch.randn_like(data) * 0.3  # augmentation 2
        z1, z2 = encoder(x1), encoder(x2)
        p1 = predictor(z1)
        if use_stop_grad:
            loss = -F.cosine_similarity(p1, z2.detach()).mean()  # stop-grad on z2
        else:
            loss = -F.cosine_similarity(p1, z2).mean()  # NO stop-grad
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return encoder(data).numpy()

z_with_sg = train_simsiam(use_stop_grad=True)
z_without_sg = train_simsiam(use_stop_grad=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.scatter(z_with_sg[:, 0], z_with_sg[:, 1], s=3, alpha=0.5, c='steelblue')
ax1.set_title('WITH stop-gradient: representations spread out', fontsize=12)
ax1.set_xlim(-5, 5); ax1.set_ylim(-5, 5); ax1.set_aspect('equal')

ax2.scatter(z_without_sg[:, 0], z_without_sg[:, 1], s=3, alpha=0.5, c='red')
ax2.set_title('WITHOUT stop-gradient: COLLAPSE! All points same location', fontsize=12)
ax2.set_xlim(-5, 5); ax2.set_ylim(-5, 5); ax2.set_aspect('equal')

plt.suptitle('SimSiam: Stop-Gradient Prevents Collapse', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Std WITH sg:    {z_with_sg.std():.4f}')
print(f'Std WITHOUT sg: {z_without_sg.std():.6f}  (near zero = collapsed!)')

### 3.3 DINO: Self-Distillation with No Labels

> 📖 **Self-study section** — covered briefly in the tutor-led recap; full detail for home reading.

DINO (Caron et al., 2021) uses **self-distillation** -- a student learns from a teacher that is itself a momentum-updated copy of the student.

- **Student** sees both small crops (local views) and large crops (global views)
- **Teacher** (momentum-updated) sees only large crops (global views)
- Loss: **cross-entropy** between softmax outputs:

$$\mathcal{L}_{\text{DINO}} = -\sum_s P_t^{(s)} \log P_s^{(s)}$$

where $P_t = \text{softmax}((g_t(x) - c) / \tau_t)$ and $P_s = \text{softmax}(g_s(x) / \tau_s)$.

**Centering** prevents collapse: $c \leftarrow m \cdot c + (1-m) \cdot \bar{g}_t$. Without centering, the teacher's softmax could become uniform or one-hot.

**Sharpening**: Teacher uses lower temperature $\tau_t < \tau_s$ for sharper distributions.

**Remarkable result**: DINO with ViT produces **attention maps that segment objects** without any segmentation supervision. The `[CLS]` token's attention naturally highlights foreground objects.

### Non-Contrastive Methods Compared

| Aspect | BYOL | SimSiam | DINO |
|---|---|---|---|
| Teacher/target | Momentum encoder | Same encoder (stop-grad) | Momentum encoder |
| Predictor | Yes | Yes | No (softmax outputs) |
| Loss | Cosine similarity | Cosine + stop-grad | Cross-entropy |
| Anti-collapse | Predictor + momentum | **Stop-gradient** | **Centering** + sharpening |

> **Transition**: Non-contrastive methods show that negatives are not fundamental -- the key is preventing collapse through architectural asymmetry. All methods so far operate within a **single modality** (images). But the richest supervisory signal may come from **different modalities** -- images and their textual descriptions. This is where CLIP enters.

---

## Phase 4: Multi-Modal Self-Supervision -- CLIP Revisited

Week 9 introduced CLIP as part of Vision-Language Models. Here we see it through the **SSL lens**:

CLIP = **cross-modal InfoNCE**. The positive pair is (image, matching text). The negatives are all non-matching pairs in the batch.

$$\mathcal{L}_{\text{CLIP}} = -\frac{1}{2N}\sum_{i=1}^{N}\left[\log\frac{e^{\text{sim}(v_i, t_i)/\tau}}{\sum_j e^{\text{sim}(v_i, t_j)/\tau}} + \log\frac{e^{\text{sim}(t_i, v_i)/\tau}}{\sum_j e^{\text{sim}(t_i, v_j)/\tau}}\right]$$

**Connecting the dots**:
- **SimCLR**: two augmented views of the same image $\to$ InfoNCE **within** modality
- **CLIP**: image + text description $\to$ InfoNCE **across** modalities
- The loss is structurally **identical** -- what changes is the source of positive pairs

CLIP also needs large batches (32,768) for the same reason as SimCLR: it is contrastive, so more negatives = better.

In [ ]:
# ── CLIP = cross-modal InfoNCE ──────────────────────────────────────────────
# Demonstrate that CLIP loss is just InfoNCE applied across modalities

def clip_loss(image_emb, text_emb, temperature=0.07):
    """CLIP symmetric contrastive loss = cross-modal InfoNCE."""
    logits = torch.matmul(image_emb, text_emb.T) / temperature
    labels = torch.arange(len(image_emb))
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

# Compare: CLIP loss on random embeddings
torch.manual_seed(42)
N, D = 8, 64
img_emb = F.normalize(torch.randn(N, D), dim=1)
txt_emb = F.normalize(torch.randn(N, D), dim=1)

loss = clip_loss(img_emb, txt_emb)
print(f'CLIP loss (random embeddings): {loss.item():.4f}')

# Make matching pairs similar
txt_emb_aligned = F.normalize(img_emb + torch.randn(N, D) * 0.1, dim=1)
loss_aligned = clip_loss(img_emb, txt_emb_aligned)
print(f'CLIP loss (aligned pairs):     {loss_aligned.item():.4f}')
print('\nCLIP IS InfoNCE across modalities. Same loss, different positive pair source.')

---

## 5. Grand Comparison

### 5.1 All Nine Methods

| Method | Phase | Supervision | Negatives | Anti-Collapse | Key Formula |
|---|---|---|---|---|---|
| **AE** | Generative | Reconstruct input | None | Bottleneck | $\|x - g(f(x))\|^2$ |
| **MAE** | Generative | Reconstruct masked patches | None | High mask ratio | MSE on masked |
| **InfoNCE** | Contrastive | (Loss function) | Required | Repulsion term | $-\log\frac{e^{s^+/\tau}}{\sum e^{s_j/\tau}}$ |
| **SimCLR** | Contrastive | Augmentation invariance | In-batch | Repulsion | Symmetric InfoNCE |
| **MoCo** | Contrastive | Augmentation invariance | Queue | Repulsion | InfoNCE + queue |
| **BYOL** | Non-contrastive | Augmentation invariance | **None** | Predictor + momentum | Cosine similarity |
| **SimSiam** | Non-contrastive | Augmentation invariance | **None** | **Stop-gradient** | Cosine + sg |
| **DINO** | Non-contrastive | Self-distillation | **None** | **Centering** | Cross-entropy |
| **CLIP** | Multi-modal | Image-text alignment | In-batch | Repulsion | Symmetric InfoNCE |

### 5.2 The Simplification Trajectory

```
SimCLR:  Large batch + negatives         (expensive)
MoCo:    Normal batch + queue             (decouple negatives from batch)
BYOL:    No negatives + momentum          (remove negatives entirely)
SimSiam: No negatives + no momentum       (simplest possible architecture)
```

### 5.3 Anti-Collapse Evolution

```
Bottleneck (AE)
   ↓
Negative repulsion (SimCLR, MoCo, CLIP)
   ↓
Predictor + Momentum (BYOL)
   ↓
Stop-gradient alone (SimSiam)
   ↓
Centering + Sharpening (DINO)
```

---
# Part C · Exam-Style Questions

Three medium-hard short-answer questions. Each intentionally pulls in concepts from earlier weeks (softmax / cross-entropy, gradient flow, BERT's MLM). Attempt each on paper first; answer sketches are in collapsed cells directly below each question — expand them only after your own attempt.

### Q1 — Temperature in InfoNCE

The InfoNCE loss is

$$\mathcal{L}_{\text{InfoNCE}} = -\log \frac{\exp(\text{sim}(q, k^+)/\tau)}{\exp(\text{sim}(q, k^+)/\tau) + \sum_{j=1}^{K}\exp(\text{sim}(q, k^-_j)/\tau)}.$$

**(a)** Show that this loss is mathematically equivalent to a $(K+1)$-way softmax cross-entropy, and identify what plays the role of "logits" and "label".

**(b)** Analyse the two extremes of temperature:
- What happens to the softmax distribution and to the gradient w.r.t. the query when $\tau \to 0$?
- What happens when $\tau \to \infty$?
- For each extreme, explain why representation learning fails.

**(c)** SimCLR uses $\tau \approx 0.1$; CLIP uses $\tau \approx 0.07$ (and actually *learns* $\tau$ during training). Give one reason CLIP can afford a slightly sharper temperature than SimCLR.

<details><summary><b>▸ Answer sketch — Q1</b></summary>

**(a)** Define the logit vector $\ell \in \mathbb{R}^{K+1}$ with entries $\ell_0 = \text{sim}(q, k^+)/\tau$ and $\ell_j = \text{sim}(q, k_j^-)/\tau$ for $j = 1, \dots, K$. Then
$$\mathcal{L}_{\text{InfoNCE}} = -\log \frac{\exp(\ell_0)}{\sum_{j=0}^{K}\exp(\ell_j)} = -\log \text{softmax}(\ell)_0 = \text{CE}(\ell, \text{label}=0).$$
So InfoNCE is a $(K+1)$-way cross-entropy classification problem where the **logits** are the scaled similarities between the query and all keys, and the **label** is the index of the positive key (by convention, index 0). This is exactly why you can implement NT-Xent in one line with `F.cross_entropy(logits, labels)` — it *is* cross-entropy.

**(b)**
- **$\tau \to 0$:** Logits $\ell_j = s_j/\tau \to \pm\infty$ in magnitude. The softmax becomes a one-hot distribution concentrated on whichever key has the single highest similarity (usually the positive, once training is underway, but at init this is essentially random). The loss gradient $\partial \mathcal{L}/\partial q = -(k^+ - \sum_j p_j k_j)/\tau$ has its softmax weights $p_j$ concentrated on one key, so only the **single hardest negative** contributes a non-trivial gradient at each step. Training becomes extremely noisy and very sensitive to which negative happens to be the current "hardest" — representations can oscillate instead of converging.
- **$\tau \to \infty$:** All logits collapse to zero, softmax becomes uniform ($1/(K+1)$), and the loss saturates at $\log(K+1)$. The gradient $\partial \mathcal{L}/\partial q$ is a *uniform* weighted sum over positives and negatives — the positive and negative contributions approximately cancel, so there is essentially no learning signal. The encoder cannot distinguish the correct pair from random pairs.
- In both extremes, learning stalls: $\tau \to 0$ produces high-variance, sparse gradients; $\tau \to \infty$ produces near-zero gradients. Practical temperatures sit in the middle where the softmax is sharp enough to carry a meaningful gradient on the positive *and* the hardest few negatives, but smooth enough that it is not dominated by a single outlier.

**(c)** Cross-modal contrast (CLIP) and within-modal contrast (SimCLR) differ in **signal-to-noise ratio**. For SimCLR, the positive pair is two augmented views of the *same image*; two different images can still be visually similar, so the gap between positives and negatives in embedding space is relatively small — a sharper temperature would over-amplify borderline negatives into noisy one-hot distributions. For CLIP, the positive pair is (image, its specific caption); a non-matching caption is semantically very different from the image, so the positives-vs-negatives gap is intrinsically wider. With a cleaner margin, CLIP can afford a sharper $\tau$ without falling into the $\tau \to 0$ failure mode. CLIP additionally makes $\tau$ *learnable* (clamped to a maximum of $\log(100) \approx 4.6$), so the model can self-regulate — if the logits ever become too peaked, the loss curvature itself pushes $\tau$ back up.
</details>

### Q2 — Why SimSiam Does Not Collapse

SimSiam uses **only positive pairs** and yet does not produce a collapsed (constant) representation. Its loss is

$$\mathcal{L}_{\text{SimSiam}} = -\frac{1}{2}\!\left[\frac{p_1 \cdot \text{sg}(z_2)}{\|p_1\|\,\|\text{sg}(z_2)\|} + \frac{p_2 \cdot \text{sg}(z_1)}{\|p_2\|\,\|\text{sg}(z_1)\|}\right].$$

**(a)** Describe the asymmetric architecture: which branch carries the predictor $h$, where the stop-gradient is applied, and why a *symmetric* version of the same loss would collapse.

**(b)** Give an intuitive explanation — referring to the EM-like alternating optimisation view in lecture — for why stop-gradient prevents collapse despite the absence of negatives.

**(c)** BYOL adds a **momentum encoder** on top of the predictor + stop-gradient. Is the momentum encoder solving the *same* anti-collapse problem as SimSiam's stop-gradient, or a *different* problem (e.g. stability / target quality)? Justify your answer.

<details><summary><b>▸ Answer sketch — Q2</b></summary>

**(a)** SimSiam has a **shared** encoder $f$ and **shared** projector $g$ for both branches, so the first half of each branch is identical. The asymmetry comes from a **predictor $h$** which only follows one branch: $p_1 = h(g(f(x_1)))$ on the online side, while the other branch produces $z_2 = g(f(x_2))$ with **no predictor**. Stop-gradient is applied to $z_2$ before it enters the loss — gradients flow through $p_1$ (and all the way back into the shared encoder via the online branch) but *not* through $z_2$. Both branches are symmetrised by swapping roles and averaging, so both encoders get updated, but at any given loss term one side is frozen.

If you made the loss truly symmetric — *both* branches carry the predictor *and* there is no stop-gradient — the network has no structural asymmetry and the symmetric cosine-similarity objective has a trivial global minimum at $f \equiv \text{const}$ (any constant output gives cosine similarity $= 1$ between the two branches for free). Gradient descent would find this minimum immediately. The whole point of the asymmetry is to break this trivial solution: the sg branch acts as a *fixed* regression target that the predictor must hit, and a constant target is not a free fixed-point for the predictor's optimisation.

**(b)** View the update as an EM-like alternating optimisation. The **"M step"**: holding $\text{sg}(z_2)$ fixed, optimise the predictor $h$ and the online encoder to minimise $\|p_1 - \text{sg}(z_2)\|$ — i.e. push $p_1$ toward the current target. The **"E step" (implicit)**: the shared encoder's update changes the representation $z_2$ at the *next* forward pass, which shifts the target to a new location. Because the target is **held fixed within a step** (via sg), the predictor is solving a genuine regression problem against a well-defined fixed point — not chasing its own tail. Without sg, both branches would co-adapt toward *each other*, and because they share weights the only self-consistent fixed point is the constant output — the collapse. The sg operator is the symmetry-breaking ingredient that converts a degenerate joint optimisation into a proper alternating one.

**(c)** BYOL's momentum encoder solves a **different** problem — **target stability**, not collapse. The landmark finding of Chen & He's SimSiam paper is that the stop-gradient *alone* is already sufficient to prevent collapse; the momentum encoder in BYOL is strictly an *optimisation* improvement that yields a slower-moving target and tends to give better absolute performance. Concretely:
- **Anti-collapse** is structural: the sg breaks the symmetry that makes constant output a minimum. SimSiam has no momentum encoder and still does not collapse — proof that momentum is not doing the anti-collapse job.
- **Stability** is numerical: BYOL's target parameters $\theta_k = m \theta_k + (1-m)\theta_q$ (with $m \approx 0.99$) change very slowly, giving the predictor a quasi-stationary regression target across many gradient steps. This makes training less noisy and lets BYOL work with slightly smaller batches.

Put differently: SimSiam proves you can get away without momentum; BYOL uses momentum because it trains more smoothly, not because it is structurally required. In the unified asymmetry-principle lens from lecture, stop-gradient is the *minimum* ingredient to prevent collapse; momentum is an optional *accelerator*.
</details>

### Q3 — Mask Ratio Design (MAE vs BERT)

BERT masks ~15% of input tokens; MAE masks ~75% of image patches. This 5× difference is a deliberate design choice.

**(a)** From the perspective of **information redundancy**, explain why a high mask ratio is necessary (and tractable) for images but not for natural language.

**(b)** Suppose we naïvely set MAE's mask ratio to 15% (matching BERT). Predict, with reasoning, what would happen to: (i) the difficulty of the pretext task, (ii) the quality of the learned representations, and (iii) the wall-clock training cost per epoch.

**(c)** MAE's encoder processes **only the visible 25% of patches** (the masked patches are inserted as learnable tokens only inside the lightweight decoder). Quantitatively, by what factor does this reduce the encoder's self-attention FLOPs compared with processing all patches? Why does this design choice make MAE practical at ViT-Huge scale?

<details><summary><b>▸ Answer sketch — Q3</b></summary>

**(a)** Images are **spatially redundant**. A $16 \times 16$ patch is highly predictable from its neighbours: similar textures, colours, and edges recur across nearby regions. Masking only 15% leaves 85% of the spatial context in place, so the pretext task degenerates into near-nearest-neighbour interpolation — the network can fill in masked patches using low-level local statistics without ever learning what is in the image. To force the encoder to learn *semantic* structure, enough of the image must be removed that local copy-paste is no longer sufficient; 75% masking is the threshold at which the task becomes non-trivial. Natural language is the opposite: each word is information-dense, and words at distance > 1 are often only weakly correlated. Masking 15% of words already removes enough to require full-sentence understanding to fill in the blanks (and in fact higher ratios quickly destroy grammatical context needed to predict anything at all). The ratio is tuned to each modality's intrinsic redundancy.

**(b)** With only 15% masking:
- **(i) Pretext task becomes too easy.** A masked patch is nearly always surrounded by 6–8 visible patches with very similar texture; a simple local predictor (or even bilinear interpolation) can reconstruct most of the masked content with low MSE. The network can minimise the loss without learning any high-level representation.
- **(ii) Representation quality collapses on downstream tasks.** With no pressure to model semantics, the encoder's features become low-level textural/statistical rather than object-aware — linear-probe accuracy on ImageNet would drop substantially, and fine-tuning would barely outperform a randomly initialised ViT.
- **(iii) Wall-clock cost per epoch increases — counterintuitively.** MAE's efficiency trick is that the encoder only processes the visible fraction. At 75% masking the encoder sees 25% of tokens; at 15% masking it sees 85%. Since self-attention scales as $T^2$, the encoder compute per step goes up by $(0.85 / 0.25)^2 \approx 11.6\times$. You get *worse* representations for *more* compute — the opposite of what you want.

**(c)** Self-attention FLOPs scale as $T^2$ (with a linear factor in $d$). At 75% masking, $T_{\text{vis}} = 0.25\, T_{\text{full}}$, so encoder self-attention FLOPs drop by a factor of
$$\frac{T_{\text{vis}}^2}{T_{\text{full}}^2} = 0.25^2 = \frac{1}{16}.$$
Concretely, on $224 \times 224$ images with patch size 16, $T_{\text{full}} = 196$ and $T_{\text{vis}} \approx 49$ — so the pairwise-interaction count drops from $\sim 38{,}000$ down to $\sim 2{,}400$, a ~16× reduction in the dominant encoder cost. This 16× saving is exactly what makes MAE tractable at ViT-Huge / ViT-2B scale: the encoder (which dominates parameters and compute) sees a short sequence, and the decoder (which sees the full sequence with mask tokens inserted) is deliberately **small and shallow** so that processing the full sequence there is cheap. Net effect: the marginal cost of adding mask tokens is almost entirely confined to the tiny decoder, while the huge encoder gets a 16× attention speedup — the trick that lets MAE outcompete contrastive methods in training efficiency at the largest scales.
</details>